In [2]:
import sys
import os
# 将项目根目录加入路径，以便识别 01_Environment 模块
sys.path.append(os.path.abspath('..'))

from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor

from Environment.liquidation_env import LiquidationEnv

# 初始化环境
env = LiquidationEnv()

# 使用 SB3 自带工具检查环境是否符合规范 (如果不报错就说明环境写得很完美)
check_env(env)
print("环境检查通过！")


2026-04-08 11:02:16.193066: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-08 11:02:16.371344: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775617336.453499   17768 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775617336.486379   17768 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-08 11:02:16.701540: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

环境检查通过！


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [3]:
# 使用 Monitor 包装环境以记录训练数据
log_dir = "../models/logs/"
os.makedirs(log_dir, exist_ok=True)
env = Monitor(LiquidationEnv(), log_dir)

# 评估环境与 Callback (边训练边测试)
eval_env = Monitor(LiquidationEnv(), log_dir)
eval_callback = EvalCallback(eval_env, best_model_save_path='../models/best_ppo_model/',
                             log_path=log_dir, eval_freq=2000,
                             deterministic=True, render=False)

# 初始化 PPO 模型
# MlpPolicy 适用于一维向量状态
model = PPO("MlpPolicy", env, verbose=1, 
            learning_rate=3e-4, 
            n_steps=2048, 
            batch_size=64, 
            gamma=1.0) # gamma=1.0 因为我们的 reward 是真正的资金成本，不需要时间折扣

# 开始训练 (由于是金融时序，建议步数设大一点，比如 100k 到 500k)
print("开始训练...")
model.learn(total_timesteps=100000, callback=eval_callback)

# 保存最终模型
model.save("../models/ppo_liquidation_final")
print("训练完成并保存模型！")


Using cuda device
Wrapping the env in a DummyVecEnv.


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


开始训练...
Eval num_timesteps=2000, episode_reward=-0.03 +/- 0.00
Episode length: 100.00 +/- 0.00
---------------------------------
| eval/              |          |
|    mean_ep_length  | 100      |
|    mean_reward     | -0.0328  |
| time/              |          |
|    total_timesteps | 2000     |
---------------------------------
New best mean reward!
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -0.0491  |
| time/              |          |
|    fps             | 279      |
|    iterations      | 1        |
|    time_elapsed    | 7        |
|    total_timesteps | 2048     |
---------------------------------
Eval num_timesteps=4000, episode_reward=-0.03 +/- 0.00
Episode length: 100.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 100          |
|    mean_reward          | -0.0292      |
| time/                   |              |
|  

In [4]:
# 1. 批量训练不同 lambda 的模型
lambdas = [0.1, 0.5, 2.0]
for l in lambdas:
    print(f"正在为 lambda = {l} 训练模型...")
    # 创建特定参数的环境
    env = LiquidationEnv(lam=l, reward_scale=1e-3) 
    
    # 初始化并训练
    model = PPO("MlpPolicy", env, verbose=0, learning_rate=3e-4)
    model.learn(total_timesteps=50000) # 建议至少5万步以看到明显区分
    
    # 保存为不同的文件名
    model.save(f"../models/ppo_lam_{l}")

print("所有模型训练完成！")

正在为 lambda = 0.1 训练模型...


/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/koro/anaconda3/envs/finrl/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


正在为 lambda = 0.5 训练模型...
正在为 lambda = 2.0 训练模型...
所有模型训练完成！
